**Inicio**

<h1>Sistema de Recomendación de Peliculas.</h1>

Empezamos cargando las bibiliotecas del ejercicio.

In [1]:
import pandas as pd
import numpy as np

print("Sistema de recomendación de peliculas")
print("Bibliotecas cargadas correctamente")

Sistema de recomendación de peliculas
Bibliotecas cargadas correctamente


Cargamos los datos de las peliculas, del archivo 'movies_metadata.csv'

In [2]:
df = pd.read_csv(
    './data/movies_metadata.csv',
    low_memory=False
)

Confirmamos que el set de datos se haya cargado correctamente, haciendo que cargue las primeras 3 peliculas del df.

In [3]:
df.head(3)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0


Revisamos la cantidad de filas y columnas del db.

In [4]:
df.shape

(45466, 24)

Buscamos la media general de las calificaciones recibidas del set de datos.

In [5]:
C = df['vote_average'].mean()

print("Media general de calificaciones:", C)

Media general de calificaciones: 5.618207215134184


Calculamos el mínimo de votos recibidos, para elegir aquellas con menos del valor entregado que son las que usaremos para rankear.

In [6]:
m = df['vote_count'].quantile(0.90)

print("Mínimo de votos requeridos:", m)

Mínimo de votos requeridos: 160.0


Ya con el valor de 160, elegiremso todas aquellas por abajo de este valor, ahora las filtraremos.

In [7]:
q_movies = df.copy().loc[df['vote_count'] >= m]

q_movies.shape

(4555, 24)

<h3>Crear la función de media ponderada</h3>


Ahora vamos a construir la fórmula que permitirá calificar las películas considerando tanto su promedio de estrellas como la cantidad de votos recibidos.

<h4>¿Qué hace esta función?</h4>


- v: cantidad de votos recibidos por la película.
- R: calificación promedio de la película.
- m: mínimo de votos establecido.
- C: promedio general de todas las películas.

La fórmula combina estos valores para obtener una puntuación más equilibrada.

In [8]:
def weighted_rating(x, m=m, C=C):

    v = x['vote_count']
    R = x['vote_average']

    return (v/(v+m) * R) + (m/(m+v) * C)

Ahora calculamos la puntuación de cada película.

Esto agregará una nueva columna llamada score a nuestro conjunto de películas filtradas.

In [9]:
q_movies['score'] = q_movies.apply(
    weighted_rating,
    axis=1
)

Mostramos las 15 mejores películas y Ordenamos de mayor a menor puntuación

In [10]:
q_movies = q_movies.sort_values(
    'score',
    ascending=False
)

q_movies[['title', 'vote_count', 'vote_average', 'score']].head(15)

,title,vote_count,vote_average,score
314,The Shawshank Redemption,8358.0,8.5,8.445869
834,The Godfather,6024.0,8.5,8.425439
10309,Dilwale Dulhania Le Jayenge,661.0,9.1,8.421453
12481,The Dark Knight,12269.0,8.3,8.265477
2843,Fight Club,9678.0,8.3,8.256385
292,Pulp Fiction,8670.0,8.3,8.251406
522,Schindler's List,4436.0,8.3,8.206639
23673,Whiplash,4376.0,8.3,8.205404
5481,Spirited Away,3968.0,8.3,8.196055
2211,Life Is Beautiful,3643.0,8.3,8.187171


Ya tenemos las 15 películas mejor posicionadas mediante la fórmula de media ponderada.

Construimos un sistema que:

- Analiza las calificaciones de las películas.
- Considera la cantidad de votos recibidos.
- Calcula una puntuación ponderada.
- Ordena las películas de mayor a menor puntuación.
- Presenta las 15 mejores recomendaciones generales.



In [35]:
from sklearn.metrics.pairwise import linear_kernel

In [36]:
# Crear un mapeo de títulos e índices
indices = pd.Series(
    df.index,
    index=df['title']
).drop_duplicates()

In [37]:
def get_recommendations(title):

    # Obtener el índice de la película
    idx = indices[title]

    # Calcular similitud con todas las películas
    sim_scores = linear_kernel(
        tfidf_matrix[idx],
        tfidf_matrix
    ).flatten()

    # Crear lista de índices y puntuaciones
    sim_scores = list(enumerate(sim_scores))

    # Ordenar de mayor a menor similitud
    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Obtener las 10 películas más similares
    sim_scores = sim_scores[1:11]

    # Obtener los índices de las películas
    movie_indices = [i[0] for i in sim_scores]

    # Devolver los títulos
    return df['title'].iloc[movie_indices]

In [38]:
get_recommendations('The Dark Knight Rises')

12481                                      The Dark Knight
150                                         Batman Forever
1328                                        Batman Returns
15511                           Batman: Under the Red Hood
585                                                 Batman
21194    Batman Unmasked: The Psychology of the Dark Kn...
9230                    Batman Beyond: Return of the Joker
18035                                     Batman: Year One
19792              Batman: The Dark Knight Returns, Part 1
3095                          Batman: Mask of the Phantasm
Name: title, dtype: str

In [39]:
get_recommendations('The Godfather')

1178               The Godfather: Part II
44030    The Godfather Trilogy: 1972-1990
1914              The Godfather: Part III
23126                          Blood Ties
11297                    Household Saints
34717                   Start Liquidation
10821                            Election
38030            A Mother Should Be Loved
17729                   Short Sharp Shock
26293                  Beck 28 - Familjen
Name: title, dtype: str

In [40]:
credits = pd.read_csv('./data/credits.csv')

print("Archivo cargado correctamente")
print("Dimensiones:", credits.shape)

credits.head(3)

Archivo cargado correctamente
Dimensiones: (45476, 3)


,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602


In [41]:
# Cargar conjuntos de datos adicionales

credits = pd.read_csv('./data/credits.csv')
keywords = pd.read_csv('./data/keywords.csv')

print("Archivos cargados correctamente")

Archivos cargados correctamente


In [42]:
# Eliminar algunos IDs problemáticos

df = df.drop([19730, 29503, 35587, 35803])

In [43]:
# Convertir todos los IDs a números enteros

keywords['id'] = keywords['id'].astype('int')
credits['id'] = credits['id'].astype('int')
df['id'] = df['id'].astype('int')

In [44]:
# Hacer merges entre DataFrames

df = df.merge(credits, on='id')
df = df.merge(keywords, on='id')

In [45]:
print("Dimensiones del nuevo DataFrame:", df.shape)

df.head(3)

Dimensiones del nuevo DataFrame: (46626, 27)


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,spoken_languages,status,tagline,title,video,vote_average,vote_count,cast,crew,keywords
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...","[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...","[{'id': 10090, 'name': 'board game'}, {'id': 1..."
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...","[{'id': 1495, 'name': 'fishing'}, {'id': 12392..."


In [46]:
import ast
import pandas as pd
import numpy as np

In [47]:
# Convertir cadenas de texto en listas de Python

def convert(text):
    return ast.literal_eval(text)

In [48]:
df['genres'] = df['genres'].apply(convert)

df['genres'].head()

0    [{'id': 16, 'name': 'Animation'}, {'id': 35, '...
1    [{'id': 12, 'name': 'Adventure'}, {'id': 14, '...
2    [{'id': 10749, 'name': 'Romance'}, {'id': 35, ...
3    [{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...
4                       [{'id': 35, 'name': 'Comedy'}]
Name: genres, dtype: object

In [49]:
df['genres'].head()

0    [{'id': 16, 'name': 'Animation'}, {'id': 35, '...
1    [{'id': 12, 'name': 'Adventure'}, {'id': 14, '...
2    [{'id': 10749, 'name': 'Romance'}, {'id': 35, ...
3    [{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...
4                       [{'id': 35, 'name': 'Comedy'}]
Name: genres, dtype: object

In [50]:
# Convertir cast, crew y keywords

features = ['cast', 'crew', 'keywords']

for feature in features:
    df[feature] = df[feature].apply(convert)

In [51]:
df[['cast', 'crew', 'keywords']].head(3)

,cast,crew,keywords
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...","[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...","[{'id': 10090, 'name': 'board game'}, {'id': 1..."
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...","[{'id': 1495, 'name': 'fishing'}, {'id': 12392..."


In [52]:
# Extraer los nombres de cada elemento

def get_list(x):
    if isinstance(x, list):
        names = [i['name'] for i in x]
        return names
    return []

In [53]:
# Obtener los tres actores principales

def get_top3(x):
    if isinstance(x, list):
        names = [i['name'] for i in x]
        return names[:3]
    return []

In [54]:
df['cast'] = df['cast'].apply(get_top3)

df['cast'].head()

0                  [Tom Hanks, Tim Allen, Don Rickles]
1       [Robin Williams, Jonathan Hyde, Kirsten Dunst]
2           [Walter Matthau, Jack Lemmon, Ann-Margret]
3    [Whitney Houston, Angela Bassett, Loretta Devine]
4           [Steve Martin, Diane Keaton, Martin Short]
Name: cast, dtype: object

In [55]:
# Obtener el nombre del director

def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan

In [56]:
# Extraer directores de las películas

df['director'] = df['crew'].apply(get_director)

df[['title', 'director']].head(10)

,title,director
0,Toy Story,John Lasseter
1,Jumanji,Joe Johnston
2,Grumpier Old Men,Howard Deutch
3,Waiting to Exhale,Forest Whitaker
4,Father of the Bride Part II,Charles Shyer
5,Heat,Michael Mann
6,Sabrina,Sydney Pollack
7,Tom and Huck,Peter Hewitt
8,Sudden Death,Peter Hyams
9,GoldenEye,Martin Campbell


In [57]:
# Extraer nombres de géneros y palabras clave

df['genres'] = df['genres'].apply(get_list)

df['keywords'] = df['keywords'].apply(get_list)

df[['title', 'genres', 'keywords']].head(5)

,title,genres,keywords
0,Toy Story,"[Animation, Comedy, Family]","[jealousy, toy, boy, friendship, friends, riva..."
1,Jumanji,"[Adventure, Fantasy, Family]","[board game, disappearance, based on children'..."
2,Grumpier Old Men,"[Romance, Comedy]","[fishing, best friend, duringcreditsstinger, o..."
3,Waiting to Exhale,"[Comedy, Drama, Romance]","[based on novel, interracial relationship, sin..."
4,Father of the Bride Part II,[Comedy],"[baby, midlife crisis, confidence, aging, daug..."


In [58]:
df[['title', 'director', 'cast', 'genres', 'keywords']].head(5)

,title,director,cast,genres,keywords
0,Toy Story,John Lasseter,"[Tom Hanks, Tim Allen, Don Rickles]","[Animation, Comedy, Family]","[jealousy, toy, boy, friendship, friends, riva..."
1,Jumanji,Joe Johnston,"[Robin Williams, Jonathan Hyde, Kirsten Dunst]","[Adventure, Fantasy, Family]","[board game, disappearance, based on children'..."
2,Grumpier Old Men,Howard Deutch,"[Walter Matthau, Jack Lemmon, Ann-Margret]","[Romance, Comedy]","[fishing, best friend, duringcreditsstinger, o..."
3,Waiting to Exhale,Forest Whitaker,"[Whitney Houston, Angela Bassett, Loretta Devine]","[Comedy, Drama, Romance]","[based on novel, interracial relationship, sin..."
4,Father of the Bride Part II,Charles Shyer,"[Steve Martin, Diane Keaton, Martin Short]",[Comedy],"[baby, midlife crisis, confidence, aging, daug..."


In [59]:
# Función para limpiar los nombres

def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        if isinstance(x, str):
            return str.lower(x.replace(" ", ""))
        return ''

In [60]:
# Limpiar las columnas de actores, director,
# géneros y palabras clave

features = ['cast', 'keywords', 'director', 'genres']

for feature in features:
    df[feature] = df[feature].apply(clean_data)

In [61]:
df[['title', 'director', 'cast', 'genres', 'keywords']].head(5)

,title,director,cast,genres,keywords
0,Toy Story,johnlasseter,"[tomhanks, timallen, donrickles]","[animation, comedy, family]","[jealousy, toy, boy, friendship, friends, riva..."
1,Jumanji,joejohnston,"[robinwilliams, jonathanhyde, kirstendunst]","[adventure, fantasy, family]","[boardgame, disappearance, basedonchildren'sbo..."
2,Grumpier Old Men,howarddeutch,"[waltermatthau, jacklemmon, ann-margret]","[romance, comedy]","[fishing, bestfriend, duringcreditsstinger, ol..."
3,Waiting to Exhale,forestwhitaker,"[whitneyhouston, angelabassett, lorettadevine]","[comedy, drama, romance]","[basedonnovel, interracialrelationship, single..."
4,Father of the Bride Part II,charlesshyer,"[stevemartin, dianekeaton, martinshort]",[comedy],"[baby, midlifecrisis, confidence, aging, daugh..."


In [62]:
# Combinar los metadatos de cada película

def create_soup(x):
    return (
        ' '.join(x['keywords']) + ' ' +
        ' '.join(x['cast']) + ' ' +
        x['director'] + ' ' +
        ' '.join(x['genres'])
    )

In [63]:
# Crear la metadata soup

df['soup'] = df.apply(create_soup, axis=1)

In [64]:
df[['title', 'soup']].head(5)

,title,soup
0,Toy Story,jealousy toy boy friendship friends rivalry bo...
1,Jumanji,boardgame disappearance basedonchildren'sbook ...
2,Grumpier Old Men,fishing bestfriend duringcreditsstinger oldmen...
3,Waiting to Exhale,basedonnovel interracialrelationship singlemot...
4,Father of the Bride Part II,baby midlifecrisis confidence aging daughter m...


In [65]:
from sklearn.feature_extraction.text import CountVectorizer

count = CountVectorizer(stop_words='english')

count_matrix = count.fit_transform(df['soup'])

count_matrix.shape

(46626, 83684)

In [66]:
from sklearn.metrics.pairwise import cosine_similarity

In [67]:
cosine_sim = cosine_similarity(
    count_matrix,
    dense_output=False
)

In [68]:
cosine_sim.shape

(46626, 46626)

In [69]:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

In [70]:
def get_content_recommendations(title):

    # Obtener el índice de la película
    idx = indices[title]

    # Obtener las similitudes de esa película
    sim_scores = cosine_sim[idx].toarray().flatten()

    # Ordenar las películas por similitud
    sim_scores = sorted(
        enumerate(sim_scores),
        key=lambda x: x[1],
        reverse=True
    )

    # Excluir la propia película
    sim_scores = sim_scores[1:11]

    # Obtener los índices de las películas recomendadas
    movie_indices = [i[0] for i in sim_scores]

    return df['title'].iloc[movie_indices]

In [71]:
get_content_recommendations('The Dark Knight Rises')

12541      The Dark Knight
10170        Batman Begins
18866            Last Exit
34458                 Rege
9271                Shiner
40211    Bullet to Beijing
8994        State of Grace
11472          Harsh Times
15004          Harry Brown
15105             Defendor
Name: title, dtype: str

In [72]:
get_content_recommendations('The Godfather')

1187       The Godfather: Part II
1926      The Godfather: Part III
11681    The Consequences of Love
15534             The Rain People
5285                  The Gambler
3315       ...And Justice for All
4582              The Cotton Club
18866                   Last Exit
34458                        Rege
35772      Manuscripts Don't Burn
Name: title, dtype: str